# Training Flow Visualization Notebook

这份 notebook 严格按当前训练代码的真实处理顺序来走：

1. 训练输入 `video / context_video`
2. `Grounded-SAM / SAM2 -> query priors`
3. `VGGT / CoTracker` 轨迹与几何
4. `object_pooler` 聚合 object tokens
5. `context_fuser` 拼接 fused context
6. `active tracks -> GT boxes` 监督
7. `Wan DiT` 前向与 loss dry-run

默认配置跟训练代码保持一致，目前 `track_source` 已经切到 `cotracker`。

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('/home/gaoya/Code_Video/Code_data/Code_vjepa_vggt')
NOTEBOOK_DIR = REPO_ROOT / 'code_vjepa_vggt' / 'train0419_reference' / 'AAAinfer'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print('repo_root =', REPO_ROOT)
print('notebook_dir =', NOTEBOOK_DIR)

In [ ]:
import matplotlib.pyplot as plt
from training_flow_notebook_helper import TrainingFlowInspector

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False

## 0. 选择训练配置与样本

这里默认使用当前 pybullet 训练配置。你可以只改 `CONFIG_PATH` 或 `SAMPLE_INDEX`，后面的单元格会按同一条训练链路重新出图。

In [ ]:
CONFIG_PATH = REPO_ROOT / 'code_vjepa_vggt' / 'configs' / 'train_0613pybullet_wan_lora_gpu67.yaml'
SAMPLE_INDEX = 0
DEVICE = None  # 例如 'cuda:0'

inspector = TrainingFlowInspector(CONFIG_PATH, sample_index=SAMPLE_INDEX, device=DEVICE)
inspector.describe()

## 1. Step 1: 训练输入

这一格对应训练器里的 `video` 和 `context_video`。下排额外叠了 GT boxes，方便直接看监督对象在 context clip 中的位置。

In [ ]:
fig = inspector.plot_step1_training_inputs(num_show=4)
plt.show()

## 2. Step 2: Grounded-SAM / SAM2 -> Query Priors

这一步严格对应训练器里的 `_maybe_build_query_priors()` 和 `_build_query_prior_for_sample()`。如果配置是 `grounded_text_multi`，会先走文本检测；失败时再回退到 motion prompt。

In [ ]:
fig = inspector.plot_step2_query_priors()
plt.show()

## 3. Step 3: Active Tracks 与 VGGT Geometry

这一步先跑 VGGT，再根据 `track_source` 决定 active tracks 是否切到 CoTracker。当前默认训练配置是 `track_source=cotracker`，所以第一行是最终进入 object pooler 的 active tracks。第二行保留 VGGT tracks，第三行补 VGGT dense geometry 的代表视图。

In [ ]:
fig = inspector.plot_step3_tracks_and_geometry(num_show=4)
plt.show()

## 4. Step 4: Object Pooler

这一格对应 `ObjectTubeProjector.forward()`：

- `jepa_tokens`：沿轨迹从 JEPA patch token 池化
- `latent_tokens`：沿轨迹从 context latents 池化
- `geom_tokens`：由 `(x, y, visibility, confidence)` 投影
- `vggt_geom_tokens`：如果有 world/depth，再补一支几何投影
- `object_tokens`：上述分支融合后的最终对象 token


In [ ]:
fig = inspector.plot_step4_object_pooler()
plt.show()

## 5. Step 5: Context Fuser

这一格对应 `ContextTokenFuser.forward()`。训练时真正送进 Wan DiT cross-attention 的条件不是 object tokens 本身，而是 `text tokens + 过滤/截断后的 object tokens`。

In [ ]:
fig = inspector.plot_step5_fused_context(max_tokens=96)
plt.show()

## 6. Step 6: Track Supervision

如果数据集里有 `context_boxes`，训练会把 active tracks 对齐到 GT boxes，并计算 `track_box_l1_loss` 和 `track_box_iou_loss`。这一步只看当前 active tracks，不再单独看非 active 分支。

In [ ]:
fig = inspector.plot_step6_track_supervision()
plt.show()

## 7. Step 7: Wan Forward / Loss Dry-Run

这一格不做真正训练更新，只跑一次 `trainer.forward(batch)`，把训练里实际用到的 latent mask、future mask 和 loss 指标拉出来，确认这条链路已经闭合。

In [ ]:
fig = inspector.plot_step7_wan_loss_path()
plt.show()
inspector.run_forward_dry_run()

## 8. 原始中间张量总览

如果要继续往下排查某一步，这里可以直接看当前样本对应的 shape / caption / track_source 等原始信息。

In [ ]:
artifacts = inspector.collect_artifacts()
summary = {
    'caption': artifacts.captions[0],
    'track_source': inspector.trainer.track_source,
    'sam_prior_sources': artifacts.sam_prior_sources,
    'sam_prompt_modes': artifacts.sam_prompt_modes,
    'active_track_image_hw': artifacts.active_track_image_hw,
    'context_video_shape': list(artifacts.context_videos.shape),
    'jepa_patch_tokens_shape': list(artifacts.jepa_patch_tokens.shape),
    'active_tracks_shape': list(artifacts.active_tracks.shape),
    'object_tokens_shape': list(artifacts.object_out.object_tokens.shape),
    'fused_context_shape': [list(x.shape) for x in artifacts.fused_context],
}
summary